In [11]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline, HuggingFaceEndpoint
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_classic.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
import os

os.environ["HF_HOME"] = "/Users/aryan/Downloads/hf_cache"

In [12]:
# llm = HuggingFacePipeline.from_model_id(
#     model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
#     task = "text-generation",
#     pipeline_kwargs={
#         "temperature":0.3
#     }
# )

# # llm = HuggingFaceEndpoint(
# #     repo_id="google/gemma-2-2b-it",
# #     task = "text-generation"
# # )

 
# model = ChatHuggingFace(llm=llm)


model = ChatGroq(
    model="qwen/qwen3.8-27b"
)

In [13]:
class Person(BaseModel):
    name: str = Field(description="Name of the person")
    age: int = Field(ge=18, description="Age of the person")
    city: str = Field(description="Name of the city the person belongs to")

In [14]:
parser = PydanticOutputParser(pydantic_object=Person)


template = PromptTemplate(
    template="Generate name, age and city of a fictional {place} person\n{format_instruction}",
    input_variables=["place"],
    partial_variables={"format_instruction":parser.get_format_instructions()}
)

In [21]:
prompt = template.invoke({"place": "india"})

response = model.invoke(prompt)
final_response = parser.parse(response.content)
final_response

Person(name='Aarav Sharma', age=28, city='Mumbai')

In [23]:
chain = template | model | parser

response = chain.invoke({"place":"nepal"})
response

Person(name='Sita Shrestha', age=28, city='Kathmandu')